# Amazon ML Challenge 2026 — FINAL BASELINE

Clean submission-oriented V1 pipeline.

**Frozen:** blocker definitions, 5 features, Logistic Regression, threshold `0.95`.

**Engineering:** reusable Parquet artifacts, DuckDB for large candidate operations, bounded smoke test before full inference.

In [1]:
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd

In [2]:
from pathlib import Path
import re, os, joblib
import polars as pl
import duckdb
from rapidfuzz.fuzz import ratio

DATA_DIR = Path("./dataset")
ARTIFACT_DIR = Path("./artifacts")
WORK_DIR = Path("./work")
OUTPUT_DIR = Path("./output")
S3_PART_DIR = ARTIFACT_DIR / "v1_s3_address_candidates"

for p in [ARTIFACT_DIR, WORK_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "v1_model.joblib"
THRESHOLD = 0.95
SMOKE_TEST_ROWS = 100_000

FILES = {
    "source1": DATA_DIR / "train_source1.tsv",
    "source2": DATA_DIR / "train_source2.tsv",
    "source3": DATA_DIR / "train_source3.tsv",
    "ground_truth": DATA_DIR / "train_ground_truth.tsv",
    "test1": DATA_DIR / "test_source1.tsv",
    "test2": DATA_DIR / "test_source2.tsv",
    "test3": DATA_DIR / "test_source3.tsv",
}

print("DATA_DIR:", DATA_DIR.resolve())
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())

DATA_DIR: F:\coding\amazonML2026\submissions\dataset
ARTIFACT_DIR: F:\coding\amazonML2026\submissions\artifacts


In [3]:
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("S3_PART_DIR:", S3_PART_DIR)
print("Exists:", S3_PART_DIR.exists())

if S3_PART_DIR.exists():
    parts = sorted(S3_PART_DIR.glob("part_*.parquet"))
    print("Partitions found:", len(parts))
    print("First:", parts[0] if parts else "NONE")

ARTIFACT_DIR: artifacts
S3_PART_DIR: artifacts\v1_s3_address_candidates
Exists: True
Partitions found: 64
First: artifacts\v1_s3_address_candidates\part_000.parquet


In [4]:
from pathlib import Path

print("S3_PART_DIR =", S3_PART_DIR)
print("Resolved    =", S3_PART_DIR.resolve())
print("Exists      =", S3_PART_DIR.exists())

parts_now = sorted(S3_PART_DIR.glob("part_*.parquet"))

print("Partitions  =", len(parts_now))

if parts_now:
    print("First       =", parts_now[0])
    print("Last        =", parts_now[-1])

S3_PART_DIR = artifacts\v1_s3_address_candidates
Resolved    = F:\coding\amazonML2026\submissions\artifacts\v1_s3_address_candidates
Exists      = True
Partitions  = 64
First       = artifacts\v1_s3_address_candidates\part_000.parquet
Last        = artifacts\v1_s3_address_candidates\part_063.parquet


## 1. Verify dataset

Expected seven TSV files under `DATA_DIR`. No exploratory work happens here.

In [5]:
missing = [k for k, p in FILES.items() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing dataset files: " + ", ".join(missing))
for k, p in FILES.items():
    print(f"{k:12s} -> {p}")

source1      -> dataset\train_source1.tsv
source2      -> dataset\train_source2.tsv
source3      -> dataset\train_source3.tsv
ground_truth -> dataset\train_ground_truth.tsv
test1        -> dataset\test_source1.tsv
test2        -> dataset\test_source2.tsv
test3        -> dataset\test_source3.tsv


## 2. Frozen normalization

This is the same normalization used by the V1 blocker.

In [6]:
def normalize_basic(name):
    if name is None:
        return ""
    name = str(name).lower()
    name = re.sub(r"[^\w\s]", " ", name)
    return re.sub(r"\s+", " ", name).strip()

LEGAL_SUFFIX_RE = re.compile(
    r"\b(private\s+limited|pvt\s+limited|pvt\s+ltd|private\s+ltd|"
    r"limited|ltd|incorporated|inc|corporation|corp|llc|llp|plc|gmbh|"
    r"sarl|sas|pte\s+ltd|pty\s+ltd|company)\s*$",
    re.IGNORECASE
)

def normalize_legal_suffix(name):
    if not name:
        return ""
    name = normalize_basic(name)
    previous = None
    while name != previous:
        previous = name
        name = LEGAL_SUFFIX_RE.sub("", name).strip()
    return name

## 3. Frozen V1 model

The original notebook established the model and saved operating point. This clean notebook **loads** the frozen model rather than silently retraining or changing it.

Features:

```text
name_edit_similarity
name_token_jaccard
address_edit_similarity
address_token_jaccard
country_match
```

In [6]:
FEATURES = [
    "name_edit_similarity",
    "name_token_jaccard",
    "address_edit_similarity",
    "address_token_jaccard",
    "country_match",
]

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"{MODEL_PATH} not found. Save the established V1 model there first."
    )

model = joblib.load(MODEL_PATH)
print("Loaded frozen V1 model:", MODEL_PATH)
print("Threshold:", THRESHOLD)

Loaded frozen V1 model: artifacts\v1_model.joblib
Threshold: 0.95


In [7]:

import polars
import sklearn

print("scikit-learn:", sklearn.__version__)
print("polars:", polars.__version__)
print("duckdb:", duckdb.__version__)

scikit-learn: 1.6.1
polars: 1.44.2
duckdb: 1.5.5


In [8]:
X_val = joblib.load("dataset/X_val.pkl")
y_val = joblib.load("dataset/y_val.pkl")

In [9]:
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_val: (34492, 5)
y_val: (34492,)


# VERIFICATION OF BASELINE AFTER MODEL RELOAD

In [10]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

THRESHOLD = 0.95

val_prob = model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= THRESHOLD).astype(int)

precision = precision_score(y_val, val_pred)
recall = recall_score(y_val, val_pred)
f05 = fbeta_score(y_val, val_pred, beta=0.5)

print("Loaded model verification")
print("=" * 40)
print(f"Precision : {precision:.4%}")
print(f"Recall    : {recall:.4%}")
print(f"F0.5      : {f05:.4%}")

Loaded model verification
Precision : 97.1154%
Recall    : 59.9644%
F0.5      : 86.4085%


## THRESHOLD = 0.95
```
Precision = 97.1154%
Recall    = 59.9644%
F0.5      = 86.4085%
```

## 4. Reuse downloaded test artifacts

The expensive Kaggle work is now treated as data, not notebook state.

Required artifacts:

```text
s1_address_tokens.parquet
s2_address_allowed.parquet
s2_address_candidates.parquet
s2_name_candidates.parquet
s3_address_allowed.parquet
s3_name_candidates.parquet
s3_address_candidates/part_000.parquet ... part_063.parquet
```

The old single `s3_address_candidates.parquet` is not used.

In [11]:
from pathlib import Path

print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("S3_PART_DIR:", S3_PART_DIR)
print("Exists:", S3_PART_DIR.exists())

if S3_PART_DIR.exists():
    parts = sorted(S3_PART_DIR.glob("part_*.parquet"))
    print("Partitions found:", len(parts))
    print("First:", parts[0] if parts else "NONE")

ARTIFACT_DIR: artifacts
S3_PART_DIR: artifacts\v1_s3_address_candidates
Exists: True
Partitions found: 64
First: artifacts\v1_s3_address_candidates\part_000.parquet


## 5A. Build final candidate partitions

We already have the expensive S3 address-token join as 64 disk-backed Parquet partitions.

We now create **64 final candidate partitions** using the same DuckDB hash rule that created those S3 address partitions:

- S2 = name candidates + S2 address candidates
- S3 = name candidates + the matching S3 address partition
- each partition is deduplicated independently
- partitions are disjoint by `source1_entity_id`

This avoids building one giant in-memory UNION.


In [ ]:
# ============================================================
# FINAL CANDIDATE PARTITIONS
# ============================================================

import duckdb
from pathlib import Path
import shutil
import time

FINAL_PART_DIR = ARTIFACT_DIR / "final_candidate_partitions"
FINAL_PART_DIR.mkdir(parents=True, exist_ok=True)

N_PARTITIONS = 64

# IMPORTANT:
# Start a fresh DuckDB connection here. The previous connection may
# already have initialized its temporary directory.
duck = duckdb.connect()

duck.execute("SET threads=2")
duck.execute("SET memory_limit='5GB'")
duck.execute("SET preserve_insertion_order=false")
duck.execute(f"SET temp_directory='{WORK_DIR.resolve().as_posix()}'")

print("DuckDB:", duck.execute("SELECT version()").fetchone()[0])
print("threads:", duck.execute("SELECT current_setting('threads')").fetchone()[0])
print("memory:", duck.execute("SELECT current_setting('memory_limit')").fetchone()[0])
print("temp:", duck.execute("SELECT current_setting('temp_directory')").fetchone()[0])


In [ ]:
# ============================================================
# GENERATE ONE FINAL CANDIDATE PARQUET PER HASH PARTITION
# ============================================================

for part_id in range(N_PARTITIONS):

    out = FINAL_PART_DIR / f"part_{part_id:03d}.parquet"

    if out.exists():
        print(f"[{part_id+1:02d}/{N_PARTITIONS}] exists -> skip")
        continue

    start = time.time()

    # S2 address is one large Parquet file, so partition it using
    # the SAME hash expression used by the original S3 address join.
    s2_addr = S2_ADDRESS_CAND_PATH.resolve().as_posix()
    s2_name = S2_NAME_PATH.resolve().as_posix()
    s3_name = S3_NAME_PATH.resolve().as_posix()
    s3_addr = (S3_PART_DIR / f"part_{part_id:03d}.parquet").resolve().as_posix()
    out_path = out.resolve().as_posix()

    query = f"""
    COPY (
        SELECT DISTINCT source1_entity_id, match_id
        FROM (

            SELECT source1_entity_id, match_id
            FROM read_parquet('{s2_name}')
            WHERE MOD(HASH(CAST(source1_entity_id AS VARCHAR)), {N_PARTITIONS}) = {part_id}

            UNION ALL

            SELECT source1_entity_id, match_id
            FROM read_parquet('{s2_addr}')
            WHERE MOD(HASH(CAST(source1_entity_id AS VARCHAR)), {N_PARTITIONS}) = {part_id}

            UNION ALL

            SELECT source1_entity_id, match_id
            FROM read_parquet('{s3_name}')
            WHERE MOD(HASH(CAST(source1_entity_id AS VARCHAR)), {N_PARTITIONS}) = {part_id}

            UNION ALL

            SELECT source1_entity_id, match_id
            FROM read_parquet('{s3_addr}')
        )
    )
    TO '{out_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """

    duck.execute(query)

    count = duck.execute(
        f"SELECT COUNT(*) FROM read_parquet('{out_path}')"
    ).fetchone()[0]

    elapsed = time.time() - start
    size_mb = out.stat().st_size / (1024**2)

    print(
        f"[{part_id+1:02d}/{N_PARTITIONS}] "
        f"{count:,} candidates | {size_mb:,.1f} MB | {elapsed/60:.1f} min"
    )

print("Final candidate partition generation complete.")


                    TEST DATA
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
        S2 name blocking    S3 name blocking
        ~15.72M pairs       ~18.77M pairs
             │                   │
             └─────────┬─────────┘
                       │
                ADDRESS BLOCKING
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
       S2 address pairs     S3 address pairs
             │                   │
             └─────────┬─────────┘
                       ▼
              64 S3 partitions
                  ✅ DONE
                       │
                       ▼
             FINAL CANDIDATE SET
                       │
             ┌─────────┴─────────┐
             │                   │
             ▼                   ▼
      candidate_pairs.tsv    MATCHING PIPELINE
                              │
                              ▼
                         Enrich pairs
                              │
                              ▼
                       5 V1 FEATURES
                              │
                              ▼
                     FROZEN MODEL
                     Logistic Regression
                              │
                              ▼
                       probability
                              │
                              ▼
                       threshold ≥ 0.95
                              │
                              ▼
                    predicted matches
                              │
                              ▼
                  matching_results.tsv
                              │
                       ┌──────┴──────┐
                       ▼             ▼
                   VALIDATOR       ZIP
                                     │
                                     ▼
                                SUBMISSION

In [ ]:
# ============================================================
# CHECKPOINT: INSPECT S3 FINAL PARTITIONS
# ============================================================

from pathlib import Path
import polars as pl

FINAL_PART_DIR = ARTIFACT_DIR / "s3_final_candidate_partitions"

parts = sorted(FINAL_PART_DIR.glob("part_*.parquet"))

print("Directory:", FINAL_PART_DIR.resolve())
print("Partitions:", len(parts))

if len(parts) != 64:
    raise RuntimeError(
        f"Expected 64 final S3 partitions, found {len(parts)}"
    )

print("\nFirst partition:")
print(pl.read_parquet(parts[0]).head())

print("\nSchema:")
print(pl.read_parquet(parts[0]).schema)

print("\nRows in first partition:")
print(pl.scan_parquet(parts[0]).select(pl.len()).collect())

In [ ]:
S3_FINAL_PART_DIR = ARTIFACT_DIR / "s3_final_candidate_partitions"

In [ ]:
# ============================================================
# BUILD S2 FINAL CANDIDATE PARTITIONS
# ============================================================

import polars as pl
import time
from pathlib import Path

N_PARTITIONS = 64

S2_FINAL_PART_DIR = ARTIFACT_DIR / "s2_final_candidate_partitions"
S2_FINAL_PART_DIR.mkdir(parents=True, exist_ok=True)

print("S2 final partition directory:")
print(S2_FINAL_PART_DIR.resolve())

print("\nInputs:")
print("S2 name   :", S2_NAME_PATH.resolve())
print("S2 address:", S2_ADDRESS_CAND_PATH.resolve())

print("\nExisting S2 final partitions:",
      len(list(S2_FINAL_PART_DIR.glob("part_*.parquet"))))

In [ ]:
# ============================================================
# CREATE 64 S2 FINAL CANDIDATE PARTITIONS
# ============================================================

s2_name = pl.scan_parquet(S2_NAME_PATH)
s2_addr = pl.scan_parquet(S2_ADDRESS_CAND_PATH)

print("Starting S2 final candidate generation...")
print("Partitions:", N_PARTITIONS)

start_all = time.time()

for i in range(N_PARTITIONS):

    output_path = S2_FINAL_PART_DIR / f"part_{i:03d}.parquet"

    # Allow safe resume if the notebook is interrupted.
    if output_path.exists():
        print(f"[{i+1:02d}/{N_PARTITIONS}] already exists, skipping")
        continue

    start = time.time()

    # Same deterministic S1 partitioning scheme for S2 and S3.
    s2_name_part = (
        s2_name
        .filter(
            pl.col("source1_entity_id")
            .hash(seed=42)
            .mod(N_PARTITIONS)
            == i
        )
        .select([
            "source1_entity_id",
            "match_id"
        ])
    )

    s2_addr_part = (
        s2_addr
        .filter(
            pl.col("source1_entity_id")
            .hash(seed=42)
            .mod(N_PARTITIONS)
            == i
        )
        .select([
            "source1_entity_id",
            "match_id"
        ])
    )

    # Exact logical operation:
    #
    # S2 name candidates
    #        UNION
    # S2 address candidates
    #        ↓
    # S2 final candidates
    #
    # unique() removes duplicates where the same pair
    # was discovered by both blockers.

    final_part = (
        pl.concat([
            s2_name_part,
            s2_addr_part
        ])
        .unique()
        .collect(engine="streaming")
    )

    final_part.write_parquet(
        output_path,
        compression="zstd"
    )

    elapsed = time.time() - start
    size_mb = output_path.stat().st_size / (1024 ** 2)

    print(
        f"[{i+1:02d}/{N_PARTITIONS}] "
        f"{final_part.height:,} candidates | "
        f"{size_mb:.1f} MB | "
        f"{elapsed/60:.1f} min"
    )

elapsed_all = time.time() - start_all

print("\nS2 final candidate partition generation complete.")
print(f"Total time: {elapsed_all/60:.1f} minutes")

```
S2 final partition i ──┐
                       ├── UNION + DEDUP ──> final partition i
S3 final partition i ──┘
```

In [ ]:
# ============================================================
# BUILD GLOBAL FINAL CANDIDATE PARTITIONS
# S2 FINAL UNION S3 FINAL
# ============================================================

import polars as pl
import time

N_PARTITIONS = 64

S2_FINAL_PART_DIR = ARTIFACT_DIR / "s2_final_candidate_partitions"
S3_FINAL_PART_DIR = ARTIFACT_DIR / "s3_final_candidate_partitions"

FINAL_CANDIDATE_DIR = ARTIFACT_DIR / "final_candidate_partitions"
FINAL_CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

print("S2 directory:", S2_FINAL_PART_DIR.resolve())
print("S3 directory:", S3_FINAL_PART_DIR.resolve())
print("Output:", FINAL_CANDIDATE_DIR.resolve())

s2_parts = sorted(S2_FINAL_PART_DIR.glob("part_*.parquet"))
s3_parts = sorted(S3_FINAL_PART_DIR.glob("part_*.parquet"))

print("\nS2 partitions:", len(s2_parts))
print("S3 partitions:", len(s3_parts))

if len(s2_parts) != 64:
    raise RuntimeError(f"Expected 64 S2 partitions, found {len(s2_parts)}")

if len(s3_parts) != 64:
    raise RuntimeError(f"Expected 64 S3 partitions, found {len(s3_parts)}")

print("\nAll 128 source partitions verified.")

In [ ]:
# ============================================================
# MERGE S2 + S3 PARTITION-BY-PARTITION
# ============================================================

start_all = time.time()

for i in range(N_PARTITIONS):

    output_path = FINAL_CANDIDATE_DIR / f"part_{i:03d}.parquet"

    if output_path.exists():
        print(f"[{i+1:02d}/{N_PARTITIONS}] already exists, skipping")
        continue

    start = time.time()

    s2_path = S2_FINAL_PART_DIR / f"part_{i:03d}.parquet"
    s3_path = S3_FINAL_PART_DIR / f"part_{i:03d}.parquet"

    # Read only this partition from each source.
    s2 = pl.scan_parquet(s2_path)
    s3 = pl.scan_parquet(s3_path)

    # --------------------------------------------------------
    # GLOBAL CANDIDATE SET FOR THIS PARTITION
    #
    # S2 candidates
    #       UNION
    # S3 candidates
    #       ↓
    # remove duplicate (S1, match) pairs
    # --------------------------------------------------------

    final_part = (
        pl.concat([
            s2,
            s3
        ])
        .select([
            "source1_entity_id",
            "match_id"
        ])
        .unique()
        .collect(engine="streaming")
    )

    final_part.write_parquet(
        output_path,
        compression="zstd"
    )

    elapsed = time.time() - start
    size_mb = output_path.stat().st_size / (1024 ** 2)

    print(
        f"[{i+1:02d}/{N_PARTITIONS}] "
        f"{final_part.height:,} candidates | "
        f"{size_mb:.1f} MB | "
        f"{elapsed/60:.1f} min"
    )

elapsed_all = time.time() - start_all

print("\n" + "=" * 60)
print("GLOBAL FINAL CANDIDATE PARTITION GENERATION COMPLETE")
print("=" * 60)
print(f"Total time: {elapsed_all/60:.1f} minutes")

## 7. Build compact test entity lookups

Only the columns needed by the matcher are retained.

In [21]:
# ============================================================
# 7. TEST ENTITY LOOKUPS
# ============================================================

print("Loading test entity tables lazily...")

test_s1 = pl.scan_csv(
    FILES["test1"],
    separator="\t"
)

test_s2 = pl.scan_csv(
    FILES["test2"],
    separator="\t"
)

test_s3 = pl.scan_csv(
    FILES["test3"],
    separator="\t"
)

print("Test tables loaded.")
print("S1 rows:", test_s1.select(pl.len()).collect().item())
print("S2 rows:", test_s2.select(pl.len()).collect().item())
print("S3 rows:", test_s3.select(pl.len()).collect().item())

Loading test entity tables lazily...
Test tables loaded.
S1 rows: 1732544
S2 rows: 4887273
S3 rows: 5082316


In [22]:
# ============================================================
# COMPACT TEST LOOKUPS
# ============================================================

s1_test_lookup = (
    test_s1
    .select([
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ])
    .rename({
        "entity_id": "source1_entity_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    })
)

s2_test_lookup = (
    test_s2
    .select([
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ])
    .rename({
        "entity_id": "match_id",
        "business_name": "match_name",
        "business_address": "match_address",
        "country": "match_country"
    })
)

s3_test_lookup = (
    test_s3
    .select([
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ])
    .rename({
        "entity_id": "match_id",
        "business_name": "match_name",
        "business_address": "match_address",
        "country": "match_country"
    })
)

candidate_test_lookup = pl.concat([
    s2_test_lookup,
    s3_test_lookup
])

print("Lookup schemas ready.")

print("\nS1:")
print(s1_test_lookup.collect_schema())

print("\nCandidate:")
print(candidate_test_lookup.collect_schema())

Lookup schemas ready.

S1:
Schema({'source1_entity_id': String, 's1_name': String, 's1_address': String, 's1_country': String})

Candidate:
Schema({'match_id': String, 'match_name': String, 'match_address': String, 'match_country': String})


## 8. Exact frozen feature functions

In [23]:
# ============================================================
# 8. EXACT FROZEN V1 FEATURE FUNCTIONS
# ============================================================

from rapidfuzz.fuzz import ratio


def token_jaccard(a, b):
    if not a or not b:
        return 0.0

    a_tokens = set(a.split())
    b_tokens = set(b.split())

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def edit_similarity(a, b):
    if not a or not b:
        return 0.0

    return ratio(a, b) / 100.0


def add_v1_features(df):
    return df.with_columns([

        # -----------------------------
        # NAME
        # -----------------------------

        pl.struct(["s1_name", "match_name"])
        .map_elements(
            lambda x: edit_similarity(
                x["s1_name"],
                x["match_name"]
            ),
            return_dtype=pl.Float64
        )
        .alias("name_edit_similarity"),

        pl.struct(["s1_name", "match_name"])
        .map_elements(
            lambda x: token_jaccard(
                x["s1_name"],
                x["match_name"]
            ),
            return_dtype=pl.Float64
        )
        .alias("name_token_jaccard"),

        # -----------------------------
        # ADDRESS
        # -----------------------------

        pl.struct(["s1_address", "match_address"])
        .map_elements(
            lambda x: edit_similarity(
                x["s1_address"],
                x["match_address"]
            ),
            return_dtype=pl.Float64
        )
        .alias("address_edit_similarity"),

        pl.struct(["s1_address", "match_address"])
        .map_elements(
            lambda x: token_jaccard(
                x["s1_address"],
                x["match_address"]
            ),
            return_dtype=pl.Float64
        )
        .alias("address_token_jaccard"),

        # -----------------------------
        # COUNTRY
        # -----------------------------

        (
            pl.col("s1_country") == pl.col("match_country")
        )
        .cast(pl.Int8)
        .alias("country_match"),
    ])


print("Frozen V1 feature functions ready.")
print("Features:", FEATURES)

Frozen V1 feature functions ready.
Features: ['name_edit_similarity', 'name_token_jaccard', 'address_edit_similarity', 'address_token_jaccard', 'country_match']


# 9. SMOKE TEST

In [24]:
# ============================================================
# 9. SMOKE TEST
# ============================================================

SMOKE_TEST_PATH = (
    ARTIFACT_DIR
    / "final_candidate_partitions"
    / "part_000.parquet"
)

print("Smoke-test source:")
print(SMOKE_TEST_PATH)

if not SMOKE_TEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing smoke-test partition: {SMOKE_TEST_PATH}"
    )

smoke_candidates = (
    pl.scan_parquet(SMOKE_TEST_PATH)
    .head(SMOKE_TEST_ROWS)
)

print(
    "Smoke candidates:",
    smoke_candidates.collect().height
)

Smoke-test source:
artifacts\final_candidate_partitions\part_000.parquet
Smoke candidates: 100000


In [26]:
# ============================================================
# SMOKE TEST: ENTITY ENRICHMENT
# ============================================================
import time
start = time.time()

smoke_enriched = (
    smoke_candidates
    .join(
        s1_test_lookup,
        on="source1_entity_id",
        how="left"
    )
    .join(
        candidate_test_lookup,
        on="match_id",
        how="left"
    )
    .collect(engine="streaming")
)

elapsed = time.time() - start

print("Enriched rows:", smoke_enriched.height)
print("Time:", f"{elapsed:.2f} seconds")

print("\nColumns:")
print(smoke_enriched.columns)

print("\nNull check:")
print(
    smoke_enriched.select([
        pl.col("s1_name").is_null().sum().alias("missing_s1_name"),
        pl.col("match_name").is_null().sum().alias("missing_match_name"),
        pl.col("s1_address").is_null().sum().alias("missing_s1_address"),
        pl.col("match_address").is_null().sum().alias("missing_match_address"),
        pl.col("s1_country").is_null().sum().alias("missing_s1_country"),
        pl.col("match_country").is_null().sum().alias("missing_match_country"),
    ])
)

Enriched rows: 100000
Time: 2.26 seconds

Columns:
['source1_entity_id', 'match_id', 's1_name', 's1_address', 's1_country', 'match_name', 'match_address', 'match_country']

Null check:
shape: (1, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ missing_s1_nam ┆ missing_match_ ┆ missing_s1_add ┆ missing_match ┆ missing_s1_co ┆ missing_match │
│ e              ┆ name           ┆ ress           ┆ _address      ┆ untry         ┆ _country      │
│ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
│ u32            ┆ u32            ┆ u32            ┆ u32           ┆ u32           ┆ u32           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
│ 0              ┆ 0              ┆ 0              ┆ 3             ┆ 0             ┆ 0             │
└────────────────┴────────────────┴────────────────┴───────────────┴───────────────┴──────────

In [27]:
# ============================================================
# SMOKE TEST: FEATURES
# ============================================================

start = time.time()

smoke_features = add_v1_features(
    smoke_enriched
)

elapsed = time.time() - start

print("Feature generation time:", f"{elapsed:.2f} seconds")

print("\nFeature sample:")
print(
    smoke_features
    .select([
        "source1_entity_id",
        "match_id",
        *FEATURES
    ])
    .head(10)
)

Feature generation time: 1.53 seconds

Feature sample:
shape: (10, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ source1_enti ┆ match_id    ┆ name_edit_s ┆ name_token_ ┆ address_edi ┆ address_tok ┆ country_mat │
│ ty_id        ┆ ---         ┆ imilarity   ┆ jaccard     ┆ t_similarit ┆ en_jaccard  ┆ ch          │
│ ---          ┆ str         ┆ ---         ┆ ---         ┆ y           ┆ ---         ┆ ---         │
│ str          ┆             ┆ f64         ┆ f64         ┆ ---         ┆ f64         ┆ i8          │
│              ┆             ┆             ┆             ┆ f64         ┆             ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ S1-329401996 ┆ S2-15885965 ┆ 0.451613    ┆ 0.125       ┆ 0.25        ┆ 0.0         ┆ 1           │
│              ┆ 0           ┆             ┆             ┆             ┆             ┆             │
│ S1-321101426 ┆ S2-7

In [28]:
!pip install -q pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import pyarrow
print("PyArrow:", pyarrow.__version__)

PyArrow: 25.0.1


In [30]:
# ============================================================
# SMOKE TEST: FROZEN MODEL
# ============================================================

X_smoke = (
    smoke_features
    .select(FEATURES)
    .to_pandas()
)

smoke_prob = model.predict_proba(X_smoke)[:, 1]

smoke_pred = (
    smoke_prob >= THRESHOLD
).astype(int)

print("============================================")
print("SMOKE TEST MODEL INFERENCE")
print("============================================")

print("Candidates:", len(smoke_pred))
print("Threshold:", THRESHOLD)
print("Predicted matches:", smoke_pred.sum())
print(
    "Positive rate:",
    f"{smoke_pred.mean():.4%}"
)

print("\nProbability range:")
print("Min:", smoke_prob.min())
print("Max:", smoke_prob.max())
print("Mean:", smoke_prob.mean())

SMOKE TEST MODEL INFERENCE
Candidates: 100000
Threshold: 0.95
Predicted matches: 1392
Positive rate: 1.3920%

Probability range:
Min: 2.805560713266465e-05
Max: 0.9999998625900011
Mean: 0.21175272652387153


## 10. Full inference gate

The remaining full run must be **partitioned and batched**. Never `.collect()` the full candidate universe.

```text
candidate partition
→ bounded batch
→ entity enrichment
→ 5 features
→ frozen model
→ P >= 0.95
→ append positive predictions
```

Then build `matching_results.tsv` from the complete test S1 list so every S1 appears exactly once.

We deliberately do not launch the full run automatically. First measure the smoke-test throughput and inspect RAM/disk behavior.

In [ ]:
# ============================================================
# PRODUCTION INFERENCE: AUTOMATIC PARTITIONS 17 → 23
# ============================================================

import time
import polars as pl

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

START_PARTITION = 50
END_PARTITION = 63

INFERENCE_DIR = ARTIFACT_DIR / "inference_matches"
INFERENCE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 250_000

# ------------------------------------------------------------
# Find all final candidate partitions
# ------------------------------------------------------------

PARTITIONS = sorted(
    FINAL_PART_DIR.glob("part_*.parquet")
)

N_PARTITIONS = len(PARTITIONS)

if N_PARTITIONS == 0:
    raise FileNotFoundError(
        f"No partitions found in {FINAL_PART_DIR}"
    )

if START_PARTITION < 0 or END_PARTITION >= N_PARTITIONS:
    raise ValueError(
        f"Requested range {START_PARTITION}-{END_PARTITION} "
        f"is outside available partitions 0-{N_PARTITIONS - 1}"
    )

# ------------------------------------------------------------
# Display overall configuration
# ------------------------------------------------------------

print("=" * 70)
print("AUTOMATIC PRODUCTION INFERENCE")
print("=" * 70)

print(f"Partitions      : {START_PARTITION} → {END_PARTITION}")
print(f"Total available : {N_PARTITIONS}")
print(f"Batch size      : {BATCH_SIZE:,}")
print(f"Output directory: {INFERENCE_DIR}")

print("=" * 70)
print()

# ------------------------------------------------------------
# Overall timing
# ------------------------------------------------------------

overall_start = time.time()

completed = []
skipped = []

# ============================================================
# RUN EACH PARTITION
# ============================================================

for PARTITION_INDEX in range(
    START_PARTITION,
    END_PARTITION + 1
):

    print()
    print("#" * 70)
    print(
        f"STARTING PARTITION {PARTITION_INDEX} "
        f"({PARTITION_INDEX - START_PARTITION + 1}/"
        f"{END_PARTITION - START_PARTITION + 1})"
    )
    print("#" * 70)

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    TEST_PARTITION = PARTITIONS[PARTITION_INDEX]

    TEST_OUTPUT = (
        INFERENCE_DIR
        / f"part_{PARTITION_INDEX:03d}_matches.parquet"
    )

    print(f"Input : {TEST_PARTITION}")
    print(f"Output: {TEST_OUTPUT}")

    # --------------------------------------------------------
    # Skip already completed partitions
    # --------------------------------------------------------

    if TEST_OUTPUT.exists():

        print()
        print(
            f"SKIPPING partition {PARTITION_INDEX}: "
            f"output already exists."
        )

        skipped.append(PARTITION_INDEX)

        continue

    # --------------------------------------------------------
    # Count partition rows
    # --------------------------------------------------------

    partition_rows = (
        pl.scan_parquet(TEST_PARTITION)
        .select(pl.len())
        .collect()
        .item()
    )

    print(f"Candidates: {partition_rows:,}")
    print(f"Batch size: {BATCH_SIZE:,}")
    print()

    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------

    positive_batches = []

    start_partition = time.time()

    for offset in range(
        0,
        partition_rows,
        BATCH_SIZE
    ):

        start_batch = time.time()

        # ----------------------------------------------------
        # 1. Read bounded candidate batch
        # ----------------------------------------------------

        candidates = (
            pl.scan_parquet(TEST_PARTITION)
            .slice(offset, BATCH_SIZE)
            .collect(engine="streaming")
        )

        # ----------------------------------------------------
        # 2. Enrich S1 + candidate entities
        # ----------------------------------------------------

        enriched = (
            candidates.lazy()

            .join(
                s1_test_lookup,
                on="source1_entity_id",
                how="left"
            )

            .join(
                candidate_test_lookup,
                on="match_id",
                how="left"
            )

            .collect(engine="streaming")
        )

        # ----------------------------------------------------
        # 3. EXACT frozen V1 features
        # ----------------------------------------------------

        features = add_v1_features(enriched)

        # ----------------------------------------------------
        # 4. Keep IDs + model features
        # ----------------------------------------------------

        feature_table = features.select([
            "source1_entity_id",
            "match_id",
            *FEATURES
        ])

        # ----------------------------------------------------
        # 5. Model inference
        # ----------------------------------------------------

        X_batch = (
            feature_table
            .select(FEATURES)
            .to_pandas()
        )

        probabilities = (
            model.predict_proba(X_batch)[:, 1]
        )

        # ----------------------------------------------------
        # 6. Frozen threshold
        # ----------------------------------------------------

        positive_mask = (
            probabilities >= THRESHOLD
        )

        positive = (
            feature_table
            .filter(
                pl.Series(
                    "positive",
                    positive_mask
                )
            )
            .with_columns(
                pl.Series(
                    "probability",
                    probabilities[positive_mask]
                )
            )
        )

        if positive.height > 0:
            positive_batches.append(positive)

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        elapsed = time.time() - start_batch

        processed = min(
            offset + BATCH_SIZE,
            partition_rows
        )

        print(
            f"[{processed:,}/{partition_rows:,}] "
            f"positives={positive.height:,} "
            f"time={elapsed:.1f}s"
        )

    # --------------------------------------------------------
    # 7. Combine positive predictions
    # --------------------------------------------------------

    if positive_batches:

        partition_matches = pl.concat(
            positive_batches,
            how="vertical"
        )

    else:

        partition_matches = pl.DataFrame({
            "source1_entity_id":
                pl.Series([], dtype=pl.String),

            "match_id":
                pl.Series([], dtype=pl.String),

            "probability":
                pl.Series([], dtype=pl.Float64),
        })

    # --------------------------------------------------------
    # 8. Save partition predictions
    # --------------------------------------------------------

    partition_matches.write_parquet(
        TEST_OUTPUT,
        compression="zstd"
    )

    elapsed_total = (
        time.time() - start_partition
    )

    # --------------------------------------------------------
    # Partition report
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("PARTITION COMPLETE")
    print("-" * 70)

    print(f"Partition        : {PARTITION_INDEX}")
    print(f"Candidates       : {partition_rows:,}")
    print(
        f"Predicted matches: "
        f"{partition_matches.height:,}"
    )
    print(
        f"Time             : "
        f"{elapsed_total / 60:.2f} minutes"
    )
    print(f"Output           : {TEST_OUTPUT}")

    completed.append(PARTITION_INDEX)


# ============================================================
# FINAL REPORT
# ============================================================

overall_elapsed = time.time() - overall_start

print()
print()
print("=" * 70)
print("AUTOMATIC INFERENCE RUN COMPLETE")
print("=" * 70)

print(
    f"Requested range : "
    f"{START_PARTITION} → {END_PARTITION}"
)

print(
    f"Completed       : "
    f"{completed}"
)

print(
    f"Skipped         : "
    f"{skipped}"
)

print(
    f"Total time      : "
    f"{overall_elapsed / 60:.2f} minutes"
)

print()
print(f"Outputs saved in:")
print(INFERENCE_DIR)

print("=" * 70)

In [12]:
# ============================================================
# 12. VERIFY ALL INFERENCE PARTITIONS
# ============================================================

from pathlib import Path
import polars as pl

INFERENCE_DIR = ARTIFACT_DIR / "inference_matches"

inference_parts = sorted(
    INFERENCE_DIR.glob("part_*_matches.parquet")
)

print("=" * 70)
print("INFERENCE OUTPUT CHECK")
print("=" * 70)

print("Directory :", INFERENCE_DIR.resolve())
print("Partitions:", len(inference_parts))

if len(inference_parts) != 64:
    raise RuntimeError(
        f"Expected 64 inference outputs, found {len(inference_parts)}"
    )

expected = [
    INFERENCE_DIR / f"part_{i:03d}_matches.parquet"
    for i in range(64)
]

missing = [
    p for p in expected
    if not p.exists()
]

if missing:
    raise RuntimeError(
        f"Missing inference partitions: {missing}"
    )

print("✓ part_000 → part_063 all present")

print("\nExample:")
print(inference_parts[0])

print("\nSchema:")
print(
    pl.read_parquet(inference_parts[0])
    .schema
)

print("\nFirst rows:")
print(
    pl.read_parquet(inference_parts[0])
    .head()
)

INFERENCE OUTPUT CHECK
Directory : F:\coding\amazonML2026\submissions\artifacts\inference_matches
Partitions: 64
✓ part_000 → part_063 all present

Example:
artifacts\inference_matches\part_000_matches.parquet

Schema:
Schema({'source1_entity_id': String, 'match_id': String, 'name_edit_similarity': Float64, 'name_token_jaccard': Float64, 'address_edit_similarity': Float64, 'address_token_jaccard': Float64, 'country_match': Int8, 'probability': Float64})

First rows:
shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ source1_en ┆ match_id   ┆ name_edit_ ┆ name_toke ┆ address_e ┆ address_t ┆ country_m ┆ probabili │
│ tity_id    ┆ ---        ┆ similarity ┆ n_jaccard ┆ dit_simil ┆ oken_jacc ┆ atch      ┆ ty        │
│ ---        ┆ str        ┆ ---        ┆ ---       ┆ arity     ┆ ard       ┆ ---       ┆ ---       │
│ str        ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ i8        ┆ f64       │
│         

In [13]:
# ============================================================
# 13. AGGREGATE PREDICTED MATCHES
# ============================================================

predicted_match_parts = []

for path in inference_parts:

    part = pl.read_parquet(path)

    predicted_match_parts.append(part)

predicted_matches = pl.concat(
    predicted_match_parts,
    how="vertical"
)

print("=" * 70)
print("PREDICTED MATCHES AGGREGATED")
print("=" * 70)

print("Rows:", predicted_matches.height)

print("\nColumns:")
print(predicted_matches.columns)

print("\nUnique S1:")
print(
    predicted_matches
    .select(pl.col("source1_entity_id").n_unique())
    .item()
)

print("\nUnique pairs:")
print(
    predicted_matches
    .select(
        pl.struct([
            "source1_entity_id",
            "match_id"
        ]).n_unique()
    )
    .item()
)

print("\nProbability statistics:")
print(
    predicted_matches
    .select([
        pl.col("probability").min().alias("min"),
        pl.col("probability").mean().alias("mean"),
        pl.col("probability").max().alias("max"),
    ])
)

PREDICTED MATCHES AGGREGATED
Rows: 18950530

Columns:
['source1_entity_id', 'match_id', 'name_edit_similarity', 'name_token_jaccard', 'address_edit_similarity', 'address_token_jaccard', 'country_match', 'probability']

Unique S1:
1635206

Unique pairs:
17215100

Probability statistics:
shape: (1, 3)
┌──────┬──────────┬─────┐
│ min  ┆ mean     ┆ max │
│ ---  ┆ ---      ┆ --- │
│ f64  ┆ f64      ┆ f64 │
╞══════╪══════════╪═════╡
│ 0.95 ┆ 0.982507 ┆ 1.0 │
└──────┴──────────┴─────┘


In [14]:
# ============================================================
# 14. DUPLICATE PREDICTION CHECK
# ============================================================

duplicate_pairs = (
    predicted_matches
    .group_by([
        "source1_entity_id",
        "match_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
)

print("=" * 70)
print("DUPLICATE PREDICTION CHECK")
print("=" * 70)

print("Duplicate pairs:", duplicate_pairs.height)

if duplicate_pairs.height > 0:
    print("\nWARNING: duplicate pairs detected!")
    print(duplicate_pairs.head(20))
    raise RuntimeError(
        "Duplicate predicted pairs detected."
    )

print("✓ No duplicate predicted pairs.")

DUPLICATE PREDICTION CHECK
Duplicate pairs: 1735430

shape: (20, 3)
┌───────────────────┬──────────────┬─────┐
│ source1_entity_id ┆ match_id     ┆ len │
│ ---               ┆ ---          ┆ --- │
│ str               ┆ str          ┆ u32 │
╞═══════════════════╪══════════════╪═════╡
│ S1-881019732      ┆ S2-150809462 ┆ 2   │
│ S1-898177027      ┆ S2-75199459  ┆ 2   │
│ S1-127468819      ┆ S2-461911186 ┆ 2   │
│ S1-472776590      ┆ S2-743067459 ┆ 2   │
│ S1-281044502      ┆ S2-812290023 ┆ 2   │
│ …                 ┆ …            ┆ …   │
│ S1-837826171      ┆ S2-427492319 ┆ 2   │
│ S1-667158130      ┆ S2-864310198 ┆ 2   │
│ S1-774918270      ┆ S2-18322695  ┆ 2   │
│ S1-526167749      ┆ S2-572175854 ┆ 2   │
│ S1-548754425      ┆ S2-52176768  ┆ 2   │
└───────────────────┴──────────────┴─────┘


RuntimeError: Duplicate predicted pairs detected.

In [15]:
# ============================================================
# 14A. DIAGNOSE DUPLICATE PREDICTIONS
# ============================================================

dup_stats = (
    predicted_matches
    .group_by([
        "source1_entity_id",
        "match_id"
    ])
    .agg([
        pl.len().alias("occurrences"),
        pl.col("probability").min().alias("prob_min"),
        pl.col("probability").max().alias("prob_max"),
        pl.col("probability").n_unique().alias("prob_unique"),
    ])
    .filter(pl.col("occurrences") > 1)
)

print("=" * 70)
print("DUPLICATE PREDICTION DIAGNOSTICS")
print("=" * 70)

print("Duplicate pair groups:", dup_stats.height)

print("\nTotal duplicate occurrences:")
print(
    dup_stats
    .select(
        (pl.col("occurrences") - 1).sum()
    )
    .item()
)

print("\nProbability differences:")
print(
    dup_stats
    .select([
        pl.col("prob_unique").max().alias("max_unique_probs"),
        (pl.col("prob_max") - pl.col("prob_min")).max()
            .alias("max_probability_difference")
    ])
)

print("\nSample duplicate groups:")
print(
    dup_stats
    .sort("occurrences", descending=True)
    .head(20)
)

DUPLICATE PREDICTION DIAGNOSTICS
Duplicate pair groups: 1735430

Total duplicate occurrences:
1735430

Probability differences:
shape: (1, 2)
┌──────────────────┬────────────────────────────┐
│ max_unique_probs ┆ max_probability_difference │
│ ---              ┆ ---                        │
│ u32              ┆ f64                        │
╞══════════════════╪════════════════════════════╡
│ 2                ┆ 2.2204e-16                 │
└──────────────────┴────────────────────────────┘

Sample duplicate groups:
shape: (20, 6)
┌───────────────────┬──────────────┬─────────────┬──────────┬──────────┬─────────────┐
│ source1_entity_id ┆ match_id     ┆ occurrences ┆ prob_min ┆ prob_max ┆ prob_unique │
│ ---               ┆ ---          ┆ ---         ┆ ---      ┆ ---      ┆ ---         │
│ str               ┆ str          ┆ u32         ┆ f64      ┆ f64      ┆ u32         │
╞═══════════════════╪══════════════╪═════════════╪══════════╪══════════╪═════════════╡
│ S1-456383424      ┆ S2-4333873

In [16]:
# ============================================================
# 14B. DEDUPLICATE PREDICTED MATCHES
# ============================================================

before_rows = predicted_matches.height

predicted_matches = (
    predicted_matches
    .unique(
        subset=[
            "source1_entity_id",
            "match_id"
        ],
        keep="first"
    )
)

after_rows = predicted_matches.height

print("=" * 70)
print("PREDICTED MATCHES DEDUPLICATED")
print("=" * 70)

print(f"Before : {before_rows:,}")
print(f"After  : {after_rows:,}")
print(f"Removed: {before_rows - after_rows:,}")

print("\nUnique S1:", predicted_matches["source1_entity_id"].n_unique())
print(
    "Unique pairs:",
    predicted_matches
    .select(
        pl.struct(["source1_entity_id", "match_id"]).n_unique()
    )
    .item()
)

PREDICTED MATCHES DEDUPLICATED
Before : 18,950,530
After  : 17,215,100
Removed: 1,735,430

Unique S1: 1635206
Unique pairs: 17215100


In [17]:
duplicate_pairs = (
    predicted_matches
    .group_by([
        "source1_entity_id",
        "match_id"
    ])
    .agg(
        pl.len().alias("count")
    )
    .filter(pl.col("count") > 1)
)

print("Remaining duplicate pairs:", duplicate_pairs.height)

Remaining duplicate pairs: 0


In [31]:
# ============================================================
# 15. VERIFY COMPLETE TEST S1 POPULATION
# ============================================================

total_test_s1 = (
    test_s1
    .select(pl.col("entity_id").n_unique())
    .collect(engine="streaming")
    .item()
)

predicted_s1 = predicted_matches["source1_entity_id"].n_unique()

print("=" * 70)
print("TEST S1 POPULATION CHECK")
print("=" * 70)

print("Total Test S1 entities :", total_test_s1)
print("S1 with predictions    :", predicted_s1)
print("S1 without predictions :", total_test_s1 - predicted_s1)

TEST S1 POPULATION CHECK
Total Test S1 entities : 1732544
S1 with predictions    : 1635206
S1 without predictions : 97338


In [37]:
# ============================================================
# 16. FINAL MATCHING RESULTS
# One row for EVERY Source 1 entity
# ============================================================

print("=" * 70)
print("BUILDING FINAL MATCHING RESULTS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Aggregate predicted matches by Source 1 entity
# ------------------------------------------------------------
#
# predicted_matches is a Polars DataFrame.
#
# Columns established in Cell 45:
#   source1_entity_id
#   match_id
#   probability
#
# Cell 48 already removed duplicate
# (source1_entity_id, match_id) pairs.
# ------------------------------------------------------------

matched_by_s1 = (
    predicted_matches
    .group_by("source1_entity_id")
    .agg(
        pl.col("match_id")
        .unique()
        .sort()
        .str.join(",")
        .alias("matched_entity_ids")
    )
)

print(
    "S1 entities with predictions:",
    matched_by_s1.height
)


# ------------------------------------------------------------
# 2. Convert the predicted-match table to LazyFrame
# ------------------------------------------------------------
#
# test_s1 is already a LazyFrame.
# Therefore both sides of the join should be LazyFrames.
# ------------------------------------------------------------

matched_by_s1_lazy = matched_by_s1.lazy()


# ------------------------------------------------------------
# 3. Build COMPLETE Source 1 population
# ------------------------------------------------------------
#
# IMPORTANT:
# test_s1 uses "entity_id", NOT "source1_entity_id".
#
# We rename it here to match the prediction table.
# ------------------------------------------------------------

test_s1_population = (
    test_s1
    .select(
        pl.col("entity_id")
        .alias("source1_entity_id")
    )
)


# ------------------------------------------------------------
# 4. LEFT JOIN predictions onto COMPLETE S1 population
# ------------------------------------------------------------
#
# This is critical.
#
# Every S1 must appear exactly once.
#
# S1 with predictions:
#     matched_entity_ids = "S2-...,S3-..."
#
# S1 without predictions:
#     matched_entity_ids = ""
# ------------------------------------------------------------

matching_results = (
    test_s1_population
    .join(
        matched_by_s1_lazy,
        on="source1_entity_id",
        how="left"
    )
    .with_columns(
        pl.col("matched_entity_ids")
        .fill_null("")
    )
    .collect(engine="streaming")
)


# ============================================================
# 5. FINAL STRUCTURAL VALIDATION
# ============================================================

expected_s1_count = (
    test_s1
    .select(
        pl.col("entity_id").n_unique()
    )
    .collect(engine="streaming")
    .item()
)

actual_rows = matching_results.height

unique_s1_count = (
    matching_results
    .select(
        pl.col("source1_entity_id").n_unique()
    )
    .item()
)

s1_with_matches = (
    matching_results
    .filter(
        pl.col("matched_entity_ids") != ""
    )
    .height
)

s1_without_matches = (
    matching_results
    .filter(
        pl.col("matched_entity_ids") == ""
    )
    .height
)


print()
print("=" * 70)
print("FINAL MATCHING RESULTS CHECK")
print("=" * 70)

print(f"Expected S1 entities : {expected_s1_count:,}")
print(f"Output rows          : {actual_rows:,}")
print(f"Unique S1 IDs        : {unique_s1_count:,}")
print(f"S1 with matches      : {s1_with_matches:,}")
print(f"S1 without matches   : {s1_without_matches:,}")


# ------------------------------------------------------------
# 6. HARD SAFETY CHECKS
# ------------------------------------------------------------

if actual_rows != expected_s1_count:
    raise RuntimeError(
        f"Row count mismatch: "
        f"expected {expected_s1_count:,}, "
        f"got {actual_rows:,}"
    )

if unique_s1_count != expected_s1_count:
    raise RuntimeError(
        "Duplicate or missing Source 1 entity IDs detected."
    )

if s1_with_matches + s1_without_matches != expected_s1_count:
    raise RuntimeError(
        "Matched + unmatched S1 counts do not equal "
        "the complete S1 population."
    )


print()
print("✓ Every Source 1 entity appears exactly once.")
print("✓ S1 entities without predictions are retained.")
print("✓ No duplicate S1 rows.")
print("✓ Final matching structure is valid.")


# ------------------------------------------------------------
# 7. Inspect sample
# ------------------------------------------------------------

print()
print("Sample:")
print(
    matching_results.head(10)
)

BUILDING FINAL MATCHING RESULTS
S1 entities with predictions: 1635206

FINAL MATCHING RESULTS CHECK
Expected S1 entities : 1,732,544
Output rows          : 1,732,544
Unique S1 IDs        : 1,732,544
S1 with matches      : 1,635,206
S1 without matches   : 97,338

✓ Every Source 1 entity appears exactly once.
✓ S1 entities without predictions are retained.
✓ No duplicate S1 rows.
✓ Final matching structure is valid.

Sample:
shape: (10, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-921369899      ┆ S2-107533354,S3-285097097,S3-3… │
│ S1-594021398      ┆ S3-133279691,S3-161627709,S3-6… │
│ S1-466641145      ┆ S3-101585900,S3-111858017,S3-1… │
│ S1-717749279      ┆ S2-161644702,S2-266731867,S2-3… │
│ S1-874436225      ┆ S2-241070484,S2-556637424,S3-2… 

In [38]:
from pathlib import Path

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MATCHING_RESULTS_PATH = OUTPUT_DIR / "matching_results.tsv"

matching_results.write_csv(
    MATCHING_RESULTS_PATH,
    separator="\t"
)

print("Saved:", MATCHING_RESULTS_PATH)
print("Exists:", MATCHING_RESULTS_PATH.exists())
print("Size:", MATCHING_RESULTS_PATH.stat().st_size / (1024**2), "MB")

Saved: output\matching_results.tsv
Exists: True
Size: 233.1594762802124 MB


## 11. Final submission contract

```text
output/
├── candidate_pairs.tsv
├── matching_results.tsv
└── validation/
```

`candidate_pairs.tsv` = final candidate set.

`matching_results.tsv` = only candidates passing the frozen `0.95` threshold.

Before submission, run the official challenge validator. Do not infer validity from notebook output alone.

### This notebook is now the tool

No exploratory experiments.
No old approaches.
No random diagnostic cells.
No fine-tuning.

Only:

```text
CONFIG
→ ARTIFACTS
→ CANDIDATES
→ FEATURES
→ FROZEN MODEL
→ OUTPUTS
→ VALIDATOR
```

In [1]:
print("hello")

hello
